In [ ]:
from PIL import Image
import os
from glob import glob
from tqdm import tqdm
import json
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import kagglehub
from google.colab import userdata


from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau,
    BackupAndRestore,
    CSVLogger,
)
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.applications import ResNet50
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
print(tf.config.list_physical_devices('GPU'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Read Data APTOS (3662)

In [ ]:
path = "/content/drive/MyDrive/Diabetic-Retinopathy/aptos_dataset/"
extract_path = "/content/drive/MyDrive/Diabetic-Retinopathy/aptos_dataset/train_full/"

In [ ]:
!find "{extract_path}" -type f | wc -l

In [ ]:
aptos_csv = pd.read_csv(os.path.join(path, "train.csv"))

In [ ]:
num_train_data = len(aptos_csv)

print(f"Jumlah train data : {num_train_data}")

In [ ]:
print("Original label distribution:")
print(aptos_csv["diagnosis"].value_counts().sort_index())

# Convert labels:
# 0 -> 0
# 1,2,3,4 -> 1
# =========================
aptos_csv["label_num"] = aptos_csv["diagnosis"].apply(
    lambda x: 0 if x == 0 else 1
)

aptos_csv["label"] = aptos_csv["diagnosis"].apply(
    lambda x: "No_DR" if x == 0 else "DR"
)

print("\nBinary label distribution:")
print(aptos_csv["label"].value_counts())

In [ ]:
def get_image_path(filename, extract_path):
    path = os.path.join(extract_path, str(filename))
    return path if os.path.exists(path) else None

aptos_csv["image_path"] = aptos_csv["id_code"].apply(lambda x: get_image_path(f"{x}.png", extract_path))

In [ ]:
print("Missing files:", aptos_csv["image_path"].isna().sum())
print("Available files:", aptos_csv["image_path"].notna().sum())

In [ ]:
aptos_csv.head()

### Read Data IDRID (455)

In [ ]:
path_idrid = "/content/drive/MyDrive/Diabetic-Retinopathy/idrid_dataset/"
extract_path_idrid = "/content/drive/MyDrive/Diabetic-Retinopathy/idrid_dataset/train_full/"

In [ ]:
!find "{extract_path_idrid}" -type f | wc -l

In [ ]:
idrid_csv = pd.read_csv(os.path.join(path_idrid, "idrid_labels.csv"))

In [ ]:
idrid_csv.columns

In [ ]:
idrid_csv = idrid_csv[['id_code', 'diagnosis', 'Risk of macular edema ']].copy()

In [ ]:
num_train_data_idrid = len(idrid_csv)

print(f"Jumlah train data : {num_train_data_idrid}")

In [ ]:
print("Original label distribution:")
print(idrid_csv["diagnosis"].value_counts().sort_index())

# Convert labels:
# 0 -> 0
# 1,2,3,4 -> 1
# =========================
idrid_csv["label_num"] = idrid_csv["diagnosis"].apply(
    lambda x: 0 if x == 0 else 1
)

idrid_csv["label"] = idrid_csv["diagnosis"].apply(
    lambda x: "No_DR" if x == 0 else "DR"
)

print("\nBinary label distribution:")
print(idrid_csv["label"].value_counts())

In [ ]:
idrid_csv["image_path"] = idrid_csv["id_code"].apply(lambda x: get_image_path(f"{x}.jpg", extract_path_idrid))

In [ ]:
idrid_csv.head()

In [ ]:
print("Missing files:", idrid_csv["image_path"].isna().sum())
print("Available files:", idrid_csv["image_path"].notna().sum())

### Read Data Messidor (1200)

In [ ]:
path_messidor = "/content/drive/MyDrive/Diabetic-Retinopathy/messidor_dataset/"
extract_path_messidor = "/content/drive/MyDrive/Diabetic-Retinopathy/messidor_dataset/train_full/"

In [ ]:
!find "{extract_path_messidor}" -type f | wc -l

In [ ]:
messidor_csv = pd.read_csv(os.path.join(path_messidor, "messidor_labels.csv"))

In [ ]:
messidor_csv.columns

In [ ]:
num_train_data_messidor = len(messidor_csv)

print(f"Jumlah train data : {num_train_data_messidor}")

In [ ]:
print("Original label distribution:")
print(messidor_csv["Retinopathy grade"].value_counts().sort_index())

# Convert labels:
# 0 -> 0
# 1,2,3,4 -> 1
# =========================
messidor_csv["label_num"] = messidor_csv["Retinopathy grade"].apply(
    lambda x: 0 if x == 0 else 1
)

messidor_csv["label"] = messidor_csv["Retinopathy grade"].apply(
    lambda x: "No_DR" if x == 0 else "DR"
)

print("\nBinary label distribution:")
print(messidor_csv["label"].value_counts())

In [ ]:
extract_path_messidor

In [ ]:
messidor_csv["image_path"] = messidor_csv["Image name"].apply(lambda x: get_image_path(x, extract_path_messidor))

In [ ]:
print("Missing files:", messidor_csv["image_path"].isna().sum())
print("Available files:", messidor_csv["image_path"].notna().sum())

### TFData APTOS Full, IDRID Full, Messidor Full

In [ ]:
train_df, val_df = train_test_split(
    aptos_csv,
    test_size=0.2,
    random_state=42,
    stratify=aptos_csv["label_num"]
)

print("Train size      :", len(train_df))
print("Validation size :", len(val_df))

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

def load_png(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = preprocess_input(image)
    return image, label

def load_jpg(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = preprocess_input(image)
    return image, label

def load_tiff(image_path, label):

    def _read_image(path):
        img = Image.open(path.decode())
        img = img.convert("RGB")
        return np.array(img)

    image = tf.numpy_function(
        _read_image,
        [image_path],
        tf.uint8
    )

    image.set_shape((None, None, 3))
    image = tf.image.resize(
        image,
        (IMG_SIZE, IMG_SIZE)
    )

    image = preprocess_input(image)

    return image, label

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices(
    (
        train_df["image_path"].values,
        train_df["label_num"].values
    )
)

train_dataset = train_dataset.map(
    load_png,
    num_parallel_calls=AUTOTUNE
)

train_dataset = train_dataset.shuffle(len(train_df))
train_dataset = train_dataset.batch(BATCH_SIZE)
train_dataset = train_dataset.prefetch(AUTOTUNE)

In [ ]:
val_dataset = tf.data.Dataset.from_tensor_slices(
    (
        val_df["image_path"].values,
        val_df["label_num"].values
    )
)

val_dataset = val_dataset.map(
    load_png,
    num_parallel_calls=AUTOTUNE
)

val_dataset = val_dataset.batch(BATCH_SIZE)
val_dataset = val_dataset.prefetch(AUTOTUNE)

In [ ]:
aptos_dataset = tf.data.Dataset.from_tensor_slices(
    (
        aptos_csv["image_path"].values,
        aptos_csv["label_num"].values
    )
)

aptos_dataset = aptos_dataset.map(
    load_png,
    num_parallel_calls=AUTOTUNE
)

aptos_dataset = aptos_dataset.batch(BATCH_SIZE)
aptos_dataset = aptos_dataset.prefetch(AUTOTUNE)

In [ ]:
idrid_dataset = tf.data.Dataset.from_tensor_slices(
    (
        idrid_csv["image_path"].values,
        idrid_csv["label_num"].values
    )
)

idrid_dataset = idrid_dataset.map(
    load_jpg,
    num_parallel_calls=AUTOTUNE
)

idrid_dataset = idrid_dataset.batch(BATCH_SIZE)
idrid_dataset = idrid_dataset.prefetch(AUTOTUNE)

In [ ]:
messidor_dataset = tf.data.Dataset.from_tensor_slices(
    (
        messidor_csv["image_path"].values,
        messidor_csv["label_num"].values
    )
)

messidor_dataset = messidor_dataset.map(
    load_tiff,
    num_parallel_calls=AUTOTUNE
)

messidor_dataset = messidor_dataset.batch(BATCH_SIZE)
messidor_dataset = messidor_dataset.prefetch(AUTOTUNE)

### Training

In [ ]:
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE,IMG_SIZE,3)
)

base_model.trainable = True

In [ ]:
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.3),
    Dense(2, activation='softmax')
])

In [ ]:
model.compile(
    optimizer=SGD(
        learning_rate=1e-4,
        momentum=0.9,
        nesterov=True
    ),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.summary()

In [ ]:
def build_callbacks(
    save_path: str,
    backup_dir: str,
    log_path: str,
    monitor: str = "val_loss",
    patience: int = 10,
):

    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    os.makedirs(backup_dir, exist_ok=True)
    os.makedirs(os.path.dirname(log_path), exist_ok=True)

    checkpoint = ModelCheckpoint(
        filepath=save_path,
        monitor=monitor,
        mode="auto",
        save_best_only=True,
        save_weights_only=False,
        verbose=1,
    )

    early_stop = EarlyStopping(
        monitor=monitor,
        mode="auto",
        patience=patience,
        min_delta=1e-4,
        restore_best_weights=True,
        verbose=1,
    )

    reduce_lr = ReduceLROnPlateau(
        monitor=monitor,
        mode="auto",
        factor=0.5,
        patience=4,
        min_delta=1e-4,
        min_lr=1e-6,
        verbose=1,
    )

    backup = BackupAndRestore(
        backup_dir=backup_dir
    )

    csv_logger = CSVLogger(
        filename=log_path,
        append=True
    )

    return [ checkpoint, early_stop, reduce_lr, backup, csv_logger]

In [ ]:
callbacks = build_callbacks(
    save_path="/content/drive/MyDrive/Diabetic-Retinopathy/V2/Model/resnet50_p_softmax.keras",
    backup_dir="/content/drive/MyDrive/Diabetic-Retinopathy/V2/Backup/resnet50_p_softmax",
    log_path="/content/drive/MyDrive/Diabetic-Retinopathy/V2/Logs/resnet50_p_softmax.csv",
    monitor="val_loss",
    patience=10,
)

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=100,
    callbacks=callbacks
)


In [ ]:
log = pd.read_csv(
    "/content/drive/MyDrive/Diabetic-Retinopathy/V2/Logs/resnet50_p_softmax.csv"
)

plt.figure(figsize=(10, 5))

plt.plot(log["epoch"] + 1, log["loss"], label="Train Loss")
plt.plot(log["epoch"] + 1, log["val_loss"], label="Validation Loss")

plt.title("Train vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.plot(log["epoch"] + 1, log["accuracy"], label="Train Accuracy")
plt.plot(log["epoch"] + 1, log["val_accuracy"], label="Validation Accuracy")

plt.title("Train vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
train_loss, train_acc = model.evaluate(train_dataset)
val_loss, val_acc = model.evaluate(val_dataset)

print("PERFORMANCE MODEL (TRAIN VAL)")

print(f"Train Accuracy      : {train_acc:.4f}")
print(f"Validation Accuracy : {val_acc:.4f}")

print(f"\nTrain Loss          : {train_loss:.4f}")
print(f"Validation Loss     : {val_loss:.4f}")

### Evaluation on APTOS

In [ ]:
y_pred_prob = model.predict(aptos_dataset)

y_pred = np.argmax(y_pred_prob, axis=1)

# true labels
y_true = aptos_csv["label_num"].values

In [ ]:
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=["No DR", "DR"], digits=4))

cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:")
print(cm)

In [ ]:
# auc scores and roc curve
y_score = y_pred_prob[:, 1]
fpr, tpr, thresholds = roc_curve(y_true, y_score)
auc_score = roc_auc_score(y_true, y_score)
print(f"ROC-AUC: {auc_score:.4f}")

In [ ]:
plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, label=f"AUC = {auc_score:.4f}")

plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve APTOS Dataset")
plt.legend()

plt.show()

### Evaluation on IDRID

In [ ]:
y_pred_prob_id = model.predict(idrid_dataset)

y_pred_id = np.argmax(y_pred_prob_id, axis=1)

# true labels
y_true_id = idrid_csv["label_num"].values

In [ ]:
print("Classification Report:")
print(classification_report(y_true_id, y_pred_id, target_names=["No DR", "DR"], digits=4))

cm_id = confusion_matrix(y_true_id, y_pred_id)
print("\nConfusion Matrix:")
print(cm_id)

In [ ]:
# auc scores and roc curve
y_score_id = y_pred_prob_id[:, 1]
fpr_id, tpr_id, thresholds_id = roc_curve(y_true_id, y_score_id)
auc_score_id = roc_auc_score(y_true_id, y_score_id)
print(f"ROC-AUC: {auc_score_id:.4f}")

In [ ]:
plt.figure(figsize=(6,6))
plt.plot(fpr_id, tpr_id, label=f"AUC = {auc_score_id:.4f}")

plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve IDRID Dataset")
plt.legend()

plt.show()

### Evaluation on Messidor

In [ ]:
y_pred_prob_m = model.predict(messidor_dataset)

y_pred_m = np.argmax(y_pred_prob_m, axis=1)

# true labels
y_true_m = messidor_csv["label_num"].values

In [ ]:
print("Classification Report:")
print(classification_report(y_true_m, y_pred_m, target_names=["No DR", "DR"], digits=4))

cm_m = confusion_matrix(y_true_m, y_pred_m)
print("\nConfusion Matrix:")
print(cm_m)

In [ ]:
# auc scores and roc curve
y_score_m = y_pred_prob_m[:, 1]
fpr_m, tpr_m, thresholds_m = roc_curve(y_true_m, y_score_m)
auc_score_m = roc_auc_score(y_true_m, y_score_m)
print(f"ROC-AUC: {auc_score_m:.4f}")

In [ ]:
plt.figure(figsize=(6,6))
plt.plot(fpr_m, tpr_m, label=f"AUC = {auc_score_m:.4f}")

plt.plot([0,1], [0,1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Messidor Dataset")
plt.legend()

plt.show()